# Stage 1: Per-Wallet Copy Sizing (tier3@2-0)

Fit the wallet universe and per-wallet copy weights ``alpha_w`` on **all data
resolved before August** (the full 2026 history), pick sizing hyperparameters
on the **val window** by sim Sharpe when one exists, single **test** pass on
**August**. When the val boundary equals the test boundary the val window is
empty: the val calibration is skipped and each scheme's default config goes
straight to test. The BUY stream is initially exposure-carved (per-wallet
capture thresholds, `EXPOSURE_FILTER_FRAC` in the setup cell), and wallet
selection runs through `signal_lab.filters` on the full (uncarved)
fit-window stats (same as `stage1_wallet_strategy_selection`). Copy qty
is capped by the reconstructed share-depth ``bucket_avail_copy_qty``.

**Output:** `stage1_scaled_result[_sport|_crypto].json` (suffix follows MARKET_FILTER_MODE; only the JSON is written)


In [29]:
# Setup: imports, paths, constants
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

NB_DIR = Path.cwd() if "__file__" not in globals() else Path(__file__).resolve().parent
sys.path.insert(0, str(NB_DIR))
OUT_DIR = NB_DIR / "signal_lab"

import numpy as np
import pandas as pd

import lib
from lib import MarketFilterMode, split_data_at_dates
from signal_lab.filters import select_strategy_eligible, select_strategy_selection
from signal_lab.signal_lib import spearman_rho
from signal_lab.sizing import (
    block_bootstrap_sharpe,
    capital_constrained_sim,
    sizing_sharpe,
)
from signal_lab.stage1 import (
    attach_copy_wallet_metrics,
    candidate_splits_for,
    load_stage1_data,
)
from signal_lab.strategies.exposure_threshold import wallet_capture_thresholds
from signal_lab.wallet_scaling import (
    alpha_kelly,
    alpha_tier,
    attach_depth_cap,
    run_sim,
    sim_row,
    wallet_daily_pnl,
    wallet_stats,
)

# ── MARKET FILTER MODE SWITCH ────────────────────────────────────────────────
MARKET_FILTER_MODE = MarketFilterMode.SPORTS
lib.MARKET_FILTER_MODE = MARKET_FILTER_MODE
DEFAULT_TAGS = lib.default_tags()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

BUDGET = 10_0000.0
EXPOSURE_BUDGET = 100000.0  # per-token (condition_id + token_id) capital cap
MAX_LEAD_DAYS = 2  # keep only trades within this many days of contract resolution
if "Politics" in DEFAULT_TAGS:
    MAX_LEAD_DAYS = 14
ALPHA_MAX_GRID = (1,)
TIER_GRID = [(nt, am, amin) for nt in (3,) for am in ALPHA_MAX_GRID for amin in (0.0, 0.25)]
UNIFORM_K_GRID = (1.0,)

# Wallet filter: strategy_selection variant follows the mode (sports mode has
# its own rules in signal_lab.filters, seeded from the weather parameters);
# the rules evaluate on the full (uncarved) fit-window stats
if MARKET_FILTER_MODE == MarketFilterMode.SPORTS:
    STRATEGY_VARIANT = "sports"
elif "Politics" in DEFAULT_TAGS:
    STRATEGY_VARIANT = "politics"
else:
    STRATEGY_VARIANT = "weather"

# Headline strategy for the exposure plot + saved result:
#   "kelly" | "tier" | "uniform" | "copy_all"  -> that scheme's scale-chosen config
#   "best"                                     -> max scale-window Sharpe overall
#                                                (copy_all when val is skipped)
SAVE_STRATEGY = "best"

# PnL variant: (pnl_col, qty_col)
PNL_VARIANT = ("copyable_pnl", "copyable_qty_5m_100")
if 'Politics' in DEFAULT_TAGS:
    PNL_VARIANT = ("copyable_pnl_20m_100", "copyable_qty_20m_100")

PNL_COL, QTY_COL = PNL_VARIANT
AVAIL_COL = QTY_COL.replace("copyable_qty_", "avail_copy_qty_")

# Windows (markets bucketed by last_condition_trade_ts):
#   fit   = everything resolved before Aug ("the whole 2026") -> wallet
#           selection (cell below recomputes metrics on this window) + alphas
#   val   = (train_end, val_end] -> sizing scheme / hyperparameter selection
#   test  = everything resolved after Aug 1 -> single honest pass
# When the val boundary equals the test boundary (train_end == val_end) the
# val window is empty: skip the val calibration (SKIP_VAL_CALIBRATION below)
# and carry each scheme's default config straight to test.
SPLIT = {"train_end": "2026-08-01", "val_end": "2026-08-01", "test_start": "2026-08-01"}
SKIP_VAL_CALIBRATION = SPLIT["train_end"] == SPLIT["val_end"]

# ── Initial exposure carve (matches stage1_wallet_strategy_selection) ────────
# The only exposure setting: right after loading, per-wallet capture
# thresholds T_{w,f} are fit on the fit-window (train + val) BUY fills via
# wallet_capture_thresholds — the smallest exposure (position * price) whose
# retained fills capture fraction `f` of the wallet's fit copyable_pnl — and
# every BUY fill (train + val + test) with exposure < T_w is dropped from the
# stream before metrics / candidate splits are computed. f == 1.0 keeps
# everything (no carve); wallets w/o positive fit copyable_pnl are kept
# whole. The carve values are exported in the saved result.
EXPOSURE_FILTER_FRAC = 0.2
EXPOSURE_PNL_COL = "copyable_pnl"
EXPOSURE_EXP_COL = "exposure"  # computed as position * price

# On-demand cleanup of the exploratory price-scale outputs (tracked in git; each run
# overwrites them via to_csv in the price-scale cells, so this is only for a clean slate):
# for _f in ("price_scale_sim.csv", "price_scale_summary.csv", "price_scale_wallets.csv"):
#     (OUT_DIR / _f).unlink(missing_ok=True)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [30]:
df_full, df_train, df_val, df_test, wallet_metrics, hold_metrics = load_stage1_data(tags=DEFAULT_TAGS, **SPLIT, max_lead_days=MAX_LEAD_DAYS)
print(f"df_full: {len(df_full):,}")
print(f"  train: {len(df_train):,}  val: {len(df_val):,}  test: {len(df_test):,}")


Markets: 4482569
Filtered markets for {'Esports'}: 318959
Loading 16 trade shards...
Total trades loaded: 23,866,579
Unique wallets: 7,722
Date range: 2025-02-08 07:19:10+00:00 -> 2026-10-14 18:48:37+00:00
Lead filter (<= 2d before resolution): 23,711,062 trades
split_data_at_dates: train_end=2026-08-01 val_end=2026-08-01 test_start=2026-08-01
  Train: 11,875,688 trades  (134,470 markets)
  Val:            0 trades  (    0 markets)
  Test:   3,932,767 trades  (28,147 markets)
  Total: 15,808,455 trades  (162,617 markets)
df_full: 15,808,455
  train: 11,875,688  val: 0  test: 3,932,767


## Copy universe

Candidate wallets = the strategy-selection filter (`signal_lab.filters.select_strategy_selection`),
fitted on the fit window (all data resolved before August) over the full
(uncarved) wallet stats — matching
`stage1_wallet_strategy_selection`.


In [31]:
# Fit window = BUY stream resolved before August (train + val).
df_fit = pd.concat([df_train, df_val])

# FULL (uncarved) fit-window wallet stats — computed before the exposure
# carve (wallet selection runs on these).
fit_metrics_full = attach_copy_wallet_metrics(df_fit)

# ── Initial exposure carve ───────────────────────────────────────────────────
# Per-wallet capture thresholds T_{w,frac} are fit on the fit-window BUY fills
# (smallest exposure whose retained fills capture EXPOSURE_FILTER_FRAC of the
# wallet's fit copyable_pnl; wallets w/o positive fit pnl get no threshold and
# are kept whole). All fills (train + val + test) with exposure
# (position * price) < T_w are dropped from the whole stream before the
# carved stats / candidate splits below are computed.
df_full[EXPOSURE_EXP_COL] = df_full["position"] * df_full["price"]
df_fit[EXPOSURE_EXP_COL] = df_fit["position"] * df_fit["price"]
exp_th = wallet_capture_thresholds(
    df_fit, (EXPOSURE_FILTER_FRAC,),
    pnl_col=EXPOSURE_PNL_COL, exp_col=EXPOSURE_EXP_COL,
)
print(f"exposure thresholds: {len(exp_th):,} wallets (frac {EXPOSURE_FILTER_FRAC:g})")

_n_full = len(df_full)
_tmap = exp_th.set_index("wallet")["threshold"]
_T = df_full["wallet"].map(_tmap).fillna(0.0)
df_full = df_full[
    df_full[EXPOSURE_EXP_COL].notna() & (df_full[EXPOSURE_EXP_COL] >= _T)
].copy()
print(f"Initial exposure carve (frac {EXPOSURE_FILTER_FRAC:g}): "
      f"{len(df_full):,} of {_n_full:,} BUY fills kept")

# Re-split the carved stream on the same date boundaries.
df_train, df_val, df_test = split_data_at_dates(
    df_full, **SPLIT, date_col="last_condition_trade_ts",
)

# Wallet selection via signal_lab.filters on the full (uncarved) stats.
eligible_wallets = select_strategy_eligible(
    fit_metrics_full, variant=STRATEGY_VARIANT,
)
wallets = select_strategy_selection(
    fit_metrics_full, variant=STRATEGY_VARIANT,
)
print(f"strategy_selection[{STRATEGY_VARIANT}] eligible (fit < {SPLIT['val_end']}): {len(eligible_wallets)}")
print(f"strategy_selection[{STRATEGY_VARIANT}] wallets (fit < {SPLIT['val_end']}): {len(wallets)}")


exposure thresholds: 5,112 wallets (frac 0.2)
Initial exposure carve (frac 0.2): 12,539,066 of 15,808,455 BUY fills kept
split_data_at_dates: train_end=2026-08-01 val_end=2026-08-01 test_start=2026-08-01
  Train:  9,313,496 trades  (117,192 markets)
  Val:            0 trades  (    0 markets)
  Test:   3,225,570 trades  (26,447 markets)
  Total: 12,539,066 trades  (143,639 markets)
strategy_selection[sports] eligible (fit < 2026-08-01): 98
strategy_selection[sports] wallets (fit < 2026-08-01): 89


In [32]:
# fit_metrics_full[fit_metrics_full['wallet'] == '0x7c63520c2ca9b336af0c205b9ccf68217bb393d4']


## Share-depth cap

Cap = stage0 Phase 2's per-bucket max copy quantity (`avail_copy_qty_5m_100`), exported with the processed trades.

In [33]:
splits = candidate_splits_for(df_full, wallets, **SPLIT)
splits = attach_depth_cap(splits, avail_col=AVAIL_COL, qty_col=QTY_COL)
for fr in splits.values():
    fr["token_group"] = fr["condition_id"] + "_" + fr["token_id"]
del df_full, df_train, df_val, df_test

for name in ("train", "val", "test"):
    fr = splits[name]
    capped = (fr["bucket_avail_copy_qty"] < fr[QTY_COL]).mean()
    print(f"{name:5s}: {len(fr):,}  trades_capped_by_depth={capped:.3f}")


split_data_at_dates: train_end=2026-08-01 val_end=2026-08-01 test_start=2026-08-01
  Train:      4,058 trades  (  586 markets)
  Val:            0 trades  (    0 markets)
  Test:       3,194 trades  (1,040 markets)
  Total:      7,252 trades  (1,626 markets)
train: 4,058  trades_capped_by_depth=0.000
val  : 0  trades_capped_by_depth=nan
test : 3,194  trades_capped_by_depth=0.000


## Fit-window per-wallet stats

Per-wallet daily pnl (copyable, alpha=1) over the full pre-August candidate
history with mean/std shrinkage -> Sharpe proxy.



In [34]:
pre_aug = pd.concat([splits["train"], splits["val"]])  # resolved before Aug
fit_daily = wallet_daily_pnl(pre_aug, pnl_col=PNL_COL)
st = wallet_stats(fit_daily, pnl_col=PNL_COL)
print(f"wallets with fit-window daily series: {len(st)}")
st[["mu", "sigma", "n_days", "sharpe_proxy", "total_pnl"]].sort_values(
    "sharpe_proxy", ascending=False
).head(15)


wallets with fit-window daily series: 89


,mu,sigma,n_days,sharpe_proxy,total_pnl
wallet,,,,,
0xccb290b1c145d1c95695d3756346bba9f1398586,29774.4313,0.0000,1,1.5999,29774.4313
0xf3ce7f04dde4f8c5aed19a20e5ecd8520e5ca57a,4105.8907,1524.7191,7,1.1442,28741.2347
0x7829d7b451c4d941704bd5f7e24d772a9097cfd3,13400.0000,0.0000,1,1.0056,13400.0000
0xf0539f60c737e58ebdebd1b56ba9474f912e3939,6951.7022,0.0000,1,0.7715,6951.7022
0xac0db77fa0f064601f5bcac4c0f1540481efb006,5413.2999,0.0000,1,0.7157,5413.2999
0x6fc3f8ead849c73df04dac7611f09b58b0b06c57,5315.7558,0.0000,1,0.7121,5315.7558
0xfc92b4a8a47231e2f8da473e945288c86c4b2e78,2486.6602,1773.5583,4,0.7092,9946.6406
0x28315c341ba19eb5fa525bf92e76ef9af4aa5473,4127.2613,0.0000,1,0.6690,4127.2613
0x540dc6de6e398f277c07f6c0b1c3136bead6122b,1037.9359,365.5913,6,0.6625,6227.6152


## Weight schemes

All benchmarked vs copy-all: shrunk max-Sharpe (Kelly), tier, uniform-k.

In [35]:
schemes = {}
for am in ALPHA_MAX_GRID:
    schemes[f"kelly@{am:g}"] = ("kelly", alpha_kelly(st, am), {"alpha_max": am})
for (nt, am, amin) in TIER_GRID:
    schemes[f"tier{nt}@{am:g}-{amin:g}"] = (
        "tier",
        alpha_tier(st, nt, am, amin),
        {"n_tiers": nt, "alpha_max": am, "alpha_min": amin},
    )
for k in UNIFORM_K_GRID:
    schemes[f"uniform@{k:g}"] = ("uniform", pd.Series(k, index=st.index), {"k": k})
schemes["copy_all"] = ("copy_all", pd.Series(1.0, index=st.index), {})

print(f"schemes: {len(schemes)}")


schemes: 5


## Scheme selection (val window)

Objective: annualized Sharpe of daily resolution-pnl, $1000 per-token exposure
cap. Best config per scheme carries into the single August test pass.

Skipped when the val boundary equals the test boundary (empty val window): each
scheme's default config — first config per family — goes straight to test with
no scale-window selection.


In [36]:
sim_rows = []
best_per_scheme = {}
skip_val = SKIP_VAL_CALIBRATION or splits["val"].empty
if skip_val:
    # Val boundary == test boundary -> empty val window: skip the val
    # calibration and carry each scheme family's default config straight to
    # test (first config per family).
    print("Val calibration skipped (val boundary == test boundary): "
          "carrying default configs, no scale-window selection.")
    for name, (scheme, alpha_map, params) in schemes.items():
        key = "kelly" if scheme == "kelly" else scheme
        if key in best_per_scheme:
            continue
        best_per_scheme[key] = (name, params, float("nan"))
else:
    for name, (scheme, alpha_map, params) in schemes.items():
        res = run_sim(splits["val"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                     group_col="token_group", group_budget=EXPOSURE_BUDGET)
        row = sim_row(scheme, name, "scale", res)
        sim_rows.append(row)
        key = scheme if scheme != "kelly" else "kelly"
        if key not in best_per_scheme or row["sharpe_daily"] > best_per_scheme[key][2]:
            best_per_scheme[key] = (name, params, row["sharpe_daily"])

sim_df = pd.DataFrame(sim_rows)
if not sim_df.empty:
    sim_df[sim_df["split"] == "scale"].sort_values("sharpe_daily", ascending=False).head(15)


Val calibration skipped (val boundary == test boundary): carrying default configs, no scale-window selection.


In [37]:
assert SAVE_STRATEGY == "best" or SAVE_STRATEGY in best_per_scheme, (
    f"SAVE_STRATEGY={SAVE_STRATEGY!r} not one of {sorted(best_per_scheme)} | 'best'"
)
if skip_val:
    print("Selected per scheme (defaults; val calibration skipped):")
    for key, (name, params, _scale_sharpe) in best_per_scheme.items():
        print(f"  {key:10s} -> {name:>22s}")
    if SAVE_STRATEGY == "best":
        # The initial exposure carve is baked into the whole stream, so the
        # 1x carved book (copy_all) is the successor of the old exposure@frac
        # headline.
        best_name = best_per_scheme["copy_all"][0]
    else:
        best_name = best_per_scheme[SAVE_STRATEGY][0]
    print(f"\nHeadline strategy ({SAVE_STRATEGY}, default): {best_name}")
else:
    print("Selected per scheme (by scale-window Sharpe):")
    for key, (name, params, scale_sharpe) in best_per_scheme.items():
        print(f"  {key:10s} -> {name:>22s}  scale_sharpe={scale_sharpe:.3f}")
    if SAVE_STRATEGY == "best":
        best_name = max(best_per_scheme.values(), key=lambda x: x[2])[0]
    else:
        best_name = best_per_scheme[SAVE_STRATEGY][0]
    print(f"\nHeadline strategy ({SAVE_STRATEGY}): {best_name}")


Selected per scheme (defaults; val calibration skipped):
  kelly      ->                kelly@1
  tier       ->              tier3@1-0
  uniform    ->              uniform@1
  copy_all   ->               copy_all

Headline strategy (best, default): copy_all


## Test: single pass per chosen config

One honest test pass for each scheme's chosen config — val-selected, or the
family default when the val calibration is skipped (10bps).

In [38]:
for key, (name, params, _scale_sharpe) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    res = run_sim(splits["test"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                 group_col="token_group", group_budget=EXPOSURE_BUDGET)
    row = sim_row(schemes[name][0], name, "test", res)
    sim_rows.append(row)

sim_df = pd.DataFrame(sim_rows)
sim_df[sim_df["split"] == "test"].sort_values("sharpe_daily", ascending=False)


,scheme,config,split,trades,pnl,roi_w,sharpe_daily,mean_used,peak_used
1,tier,tier3@1-0,test,672,134749.6900,0.0866,1.7380,1358.4800,75863.7900
2,uniform,uniform@1,test,2743,198805.4800,0.0941,1.2460,2344.4300,71140.3400
3,copy_all,copy_all,test,2743,198805.4800,0.0941,1.2460,2344.4300,71140.3400
0,kelly,kelly@1,test,2743,213763.1200,0.1058,1.1510,2173.4500,72650.5000


## Robustness: cost sweep + bootstrap CI

Cost sweep (0/10/30bps) + 7-day block-bootstrap Sharpe CI on test.

In [39]:
ci_rows = []
for key, (name, params, _) in best_per_scheme.items():
    alpha_map = schemes[name][1]
    res = run_sim(splits["test"], alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
                 group_col="token_group", group_budget=EXPOSURE_BUDGET)
    point, lo, hi = block_bootstrap_sharpe(res["daily_pnl"], block_size=7, n_iter=1000, seed=42)
    ci_rows.append({
        "design": name,
        "pnl": round(res["net_pnl"], 2),
        "roi_w": round(res["net_pnl"] / res["notional"], 4) if res["notional"] > 0 else np.nan,
        "sharpe_daily": round(sizing_sharpe(res["daily_pnl"], 365.0), 3),
        "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
    })

res_all = capital_constrained_sim(splits["test"], "score1", float("inf"), 1.0,
                                  group_col="token_group", group_budget=EXPOSURE_BUDGET,
                                  pnl_col=PNL_COL, qty_col=QTY_COL)
point, lo, hi = block_bootstrap_sharpe(res_all["daily_pnl"], block_size=7, n_iter=1000, seed=42)
ci_rows.append({
    "design": "copy_all",
    "pnl": round(res_all["net_pnl"], 2),
    "roi_w": round(res_all["net_pnl"] / res_all["notional"], 4) if res_all["notional"] > 0 else np.nan,
    "sharpe_daily": round(sizing_sharpe(res_all["daily_pnl"], 365.0), 3),
    "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
})

ci_df = pd.DataFrame(ci_rows)
ci_df


,design,pnl,roi_w,sharpe_daily,ci_lo,ci_hi
0,kelly@1,213763.1200,0.1058,1.1510,-4.8980,0.8900
1,tier3@1-0,134749.6900,0.0866,1.7380,-3.3880,4.4110
2,uniform@1,198805.4800,0.0941,1.2460,-5.3020,1.0700
3,copy_all,198805.4800,0.0941,1.2460,-5.3020,1.0700
4,copy_all,198805.4800,0.0941,1.2460,-5.3020,1.0700


## Test-period exposure & PnL over time

PnL is attributed twice per book: at trade time (`dt`) and at contract resolution time (`last_condition_trade_ts`, resolved contracts only).

The whole stream is already carved by the initial exposure carve (`EXPOSURE_FILTER_FRAC`): candidate BUYs whose per-wallet exposure `position * price` was below the fit-fitted threshold were dropped before metrics / splits.

- **sized**: the headline scheme on the carved candidate test split — run through the budget-aware sim (`qty = alpha_w * copyable_qty_20m_100`, depth-capped by `bucket_avail_copy_qty`).
- **copy_all**: reference = copy every carved candidate BUY at 1x — PnL is the raw `copyable_pnl_20m_100` column over the carved candidate BUYs.
- **wallet buy pnl**: the copy wallets' own realized PnL.

**Exposure** (trigger-wallet reference) is drawn over the same carved book as the sized PnL: exposure = `position * price` on the token, opened when the trade happened (`dt`), closed at contract resolution (`res_ts`).


In [40]:
import plotly.graph_objects as go

test = splits["test"].copy()
test["res_ts"] = pd.to_datetime(test["last_condition_trade_ts"], utc=True, errors="coerce")
test["copy_all_pnl"] = test[PNL_COL]
test["wallet_buy_pnl"] = test["pnl"]

# Sized book = the headline scheme on the candidate test split — the whole
# stream is already carved by the initial exposure carve (EXPOSURE_FILTER_FRAC).
book = splits["test"].copy()
book["res_ts"] = pd.to_datetime(book["last_condition_trade_ts"], utc=True, errors="coerce")
alpha_map = schemes[best_name][1]
book["alpha_w"] = book["wallet"].map(alpha_map).fillna(1.0)

# Run sim to get the taken mask — exposure/qty must respect the budget
res = run_sim(book, alpha_map, pnl_col=PNL_COL, qty_col=QTY_COL,
              group_col="token_group", group_budget=EXPOSURE_BUDGET)
taken_idx = set(res["taken"].values)
book["taken"] = book.index.isin(taken_idx)
book["qty"] = np.clip(book["alpha_w"] * book[QTY_COL], 0.0, book["bucket_avail_copy_qty"])
book["copy_pnl"] = np.where(
    book["taken"],
    book[PNL_COL] / book[QTY_COL].replace(0, np.nan) * book["qty"],
    0.0,
)

print(f"carved book (initial carve {EXPOSURE_FILTER_FRAC:g}): {len(book):,} trades  "
      f"taken {int(book["taken"].sum()):,}  "
      f"sim PnL: {book.loc[book["taken"], "copy_pnl"].sum():,.0f}")

window_end = test["dt"].max()
resolved = test["res_ts"] <= window_end
print(
    f"test trades: {len(test):,}  contracts: {test["condition_id"].nunique():,}  "
    f"resolved by {window_end:%Y-%m-%d}: {int(resolved.sum()):,} trades "
    f"({test.loc[resolved, "condition_id"].nunique():,} contracts)"
)
print(
    f"copy_all {PNL_COL} sum: {test[PNL_COL].sum():,.0f}  "
    f"wallet pnl sum: {test["wallet_buy_pnl"].sum():,.0f}  "
    f"sized sim copy_pnl sum: {book.loc[book["taken"], "copy_pnl"].sum():,.0f}"
)

# Exposure (trigger-wallet reference): the initial-carved book, exposure =
# position * price (EXPOSURE_EXP_COL), opened when the trade happened (dt),
# closed at contract resolution (res_ts, resolved contracts only).
ca_resolved = book["res_ts"] <= window_end
ca_ev = pd.DataFrame({
    "ev_dt": book["dt"],
    "exposure_delta": book[EXPOSURE_EXP_COL],
})
ca_cl = pd.DataFrame({
    "ev_dt": book.loc[ca_resolved, "res_ts"],
    "exposure_delta": -book.loc[ca_resolved, EXPOSURE_EXP_COL],
})
exposure_events = (
    pd.concat([ca_ev, ca_cl], ignore_index=True)
    .sort_values("ev_dt")
    .reset_index(drop=True)
)
exposure_events["exposure"] = exposure_events["exposure_delta"].cumsum()


# Bot cumulative exposure: the sized position on each token (taken qty * price),
# opened at dt, closed at contract resolution (res_ts, resolved contracts only).
bt_resolved = book["taken"] & (book["res_ts"] <= window_end)
bt_ev = pd.DataFrame({
    "ev_dt": book.loc[book["taken"], "dt"],
    "exposure_delta": book.loc[book["taken"], "qty"] * book.loc[book["taken"], "price"],
})
bt_cl = pd.DataFrame({
    "ev_dt": book.loc[bt_resolved, "res_ts"],
    "exposure_delta": -(book.loc[bt_resolved, "qty"] * book.loc[bt_resolved, "price"]),
})
bot_events = (
    pd.concat([bt_ev, bt_cl], ignore_index=True)
    .sort_values("ev_dt")
    .reset_index(drop=True)
)
bot_events["exposure"] = bot_events["exposure_delta"].cumsum()

def _cum_pnl(df, dt_col, pnl_col):
    d = df[[dt_col, pnl_col]].rename(columns={dt_col: "ev_dt", pnl_col: "pnl"})
    d = d.sort_values("ev_dt").reset_index(drop=True)
    d["cum_pnl"] = d["pnl"].cumsum()
    return d

pnl_trade = _cum_pnl(book, "dt", "copy_pnl")
pnl_res = _cum_pnl(book, "res_ts", "copy_pnl")
pnl_copy_all_trade = _cum_pnl(test, "dt", "copy_all_pnl")
pnl_copy_all_res = _cum_pnl(test, "res_ts", "copy_all_pnl")
pnl_wallet_trade = _cum_pnl(test, "dt", "wallet_buy_pnl")
pnl_wallet_res = _cum_pnl(test, "res_ts", "wallet_buy_pnl")

fig = go.Figure()
fig.add_trace(go.Scatter(x=bot_events["ev_dt"], y=bot_events["exposure"], mode="lines",
    name="bot cumulative exposure (sized book)", line=dict(color="rgba(148,103,189,0.9)")))
fig.add_trace(go.Scatter(x=exposure_events["ev_dt"], y=exposure_events["exposure"], mode="lines",
    name=f"trigger-wallet exposure (initial carve {EXPOSURE_FILTER_FRAC:g})", line=dict(color="rgba(31,119,180,0.7)")))
fig.add_trace(go.Scatter(
    x=pnl_trade["ev_dt"], y=pnl_trade["cum_pnl"], mode="lines",
    name=f"cum {PNL_COL} sized (trade time)",
))
fig.add_trace(go.Scatter(
    x=pnl_res["ev_dt"], y=pnl_res["cum_pnl"], mode="lines", line=dict(dash="dash"),
    name=f"cum {PNL_COL} sized (resolution time)",
))
fig.add_trace(go.Scatter(
    x=pnl_copy_all_trade["ev_dt"], y=pnl_copy_all_trade["cum_pnl"], mode="lines",
    name=f"cum {PNL_COL} copy_all (trade time)", line=dict(color="rgba(255,127,14,0.6)"),
))
fig.add_trace(go.Scatter(
    x=pnl_copy_all_res["ev_dt"], y=pnl_copy_all_res["cum_pnl"], mode="lines",
    name=f"cum {PNL_COL} copy_all (resolution time)", line=dict(dash="dash", color="rgba(255,127,14,0.6)"),
))
fig.add_trace(go.Scatter(
    x=pnl_wallet_trade["ev_dt"], y=pnl_wallet_trade["cum_pnl"], mode="lines",
    name="cum wallet buy pnl (trade time)", line=dict(color="rgba(44,160,28,0.6)"),
))
fig.add_trace(go.Scatter(
    x=pnl_wallet_res["ev_dt"], y=pnl_wallet_res["cum_pnl"], mode="lines",
    name="cum wallet buy pnl (resolution time)", line=dict(dash="dash", color="rgba(44,160,28,0.6)"),
))
fig.update_layout(
    title=dict(text=f"Test-period exposure & PnL over time — {best_name}", x=0.5),
    xaxis_title="Time",
    yaxis_title="USDC",
    template="plotly_white",
    margin=dict(t=80, b=110),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.25,
        xanchor="center",
        x=0.5,
    ),
)
fig.show()


carved book (initial carve 0.2): 3,194 trades  taken 2,743  sim PnL: 198,805
test trades: 3,194  contracts: 1,040  resolved by 2026-10-14: 3,187 trades (1,035 contracts)
copy_all copyable_pnl sum: 198,805  wallet pnl sum: 416,300  sized sim copy_pnl sum: 198,805


## Per-wallet contributions

Train alphas vs forward (test) wallet stats.

In [41]:
test_daily = wallet_daily_pnl(splits["test"], pnl_col=PNL_COL)
test_st = test_daily.groupby("wallet")[PNL_COL].agg(
    test_pnl="sum", test_n_days="size"
)
test_sharpe = test_daily.groupby("wallet")[PNL_COL].apply(
    lambda s: (s.mean() / s.std() * np.sqrt(365.0)) if s.std() > 0 and len(s) >= 2 else np.nan
).rename("test_sharpe")

contrib = st.join(test_st, how="outer").join(test_sharpe, how="outer").fillna(0.0)
contrib = contrib[contrib["test_n_days"] > 0]
alpha_cont = schemes[best_per_scheme["kelly"][0]][1]
alpha_tier_cont = schemes[best_per_scheme["tier"][0]][1]
contrib["alpha_kelly"] = contrib.index.map(alpha_cont).fillna(1.0)
contrib["alpha_tier"] = contrib.index.map(alpha_tier_cont).fillna(1.0)
contrib = contrib.reset_index()
contrib["test_roi"] = contrib["test_pnl"] / contrib["total_pnl"].replace(0, np.nan)

a = contrib["alpha_kelly"].to_numpy()
ts = contrib["test_sharpe"].to_numpy()
valid = np.isfinite(ts)
rho = spearman_rho(pd.Series(a[valid]), pd.Series(ts[valid])) if valid.sum() > 2 else np.nan
print(f"Spearman(alpha_kelly, wallet test sharpe) = {rho:.4f}  (n={int(valid.sum())})")
contrib[["wallet", "alpha_kelly", "alpha_tier", "test_pnl", "test_sharpe", "test_roi"]].head(15)


Spearman(alpha_kelly, wallet test sharpe) = 0.1746  (n=19)


,wallet,alpha_kelly,alpha_tier,test_pnl,test_sharpe,test_roi
0,0x08c263965a9cbec5e83ac1d6f14b9bbc957bc023,0.6814,0.9889,34369.0900,1.7035,1.9834
1,0x1ca03f0bc7af7d39a6d35c2191bcd39eebf538ba,1.0115,0.9889,-1194.5753,-5.2434,-0.3540
2,0x51d97a401324e10eada45dfbc55a1f8cc202cb1a,0.9631,0.0000,-125.8102,0.0000,-0.0978
3,0x58f8f1138be2192696378629fc9aa23c7910dc70,0.7549,0.9889,11485.5376,2.6296,1.4553
4,0x5c2311acccc5e9a172f0ce23d4480177bc00cab3,0.9905,0.9889,8670.1773,10.8481,4.8273
5,0x64f4577177d590e78174b0c5364af8e80fbab38e,1.0389,0.0000,8852.4063,1.8886,2.8250
6,0x733b2ffa4ebfd125ee71e8b673bd08273abd7260,0.9078,0.9889,-4569.5434,-1.3934,-0.5972
7,0x795173772d871733c3110a5695d9638e4b50bb41,0.7208,0.0000,4755.2461,4.8890,0.3748
8,0x7986576408b65a9805ac85553be33ad0b044baa5,1.0027,0.0000,58266.7220,4.3703,7.4007
9,0xa0e12cca77c2b18bea2449625c87bf99e295b9b4,1.1388,0.9889,-788.9013,-1.2428,-0.1895


## Save stage 1 result

In [42]:
import json
from datetime import datetime, timezone

best_params = schemes[best_name][2]

# Initial exposure carve values: per-wallet thresholds at EXPOSURE_FILTER_FRAC,
# restricted to the selected wallets (what the downstream copy bot needs).
exp_th_map = pd.Series(exp_th["threshold"].to_numpy(), index=exp_th["wallet"])
exp_th_map = exp_th_map[exp_th_map.index.isin(wallets)]
contrib["exposure_threshold"] = contrib["wallet"].map(exp_th_map)

wallet_cols = [
    "wallet", "mu", "sigma", "n_days", "total_pnl", "sharpe_proxy",
    "alpha_kelly", "alpha_tier", "test_pnl", "test_n_days", "test_sharpe", "test_roi",
    "exposure_threshold",
]
wallet_records = contrib[[c for c in wallet_cols if c in contrib.columns]].to_dict(orient="records")


def _convert(obj):
    if isinstance(obj, (float, np.floating)) and np.isnan(obj):
        return None
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

test_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == best_name)].iloc[0]
copy_all_row = sim_df[(sim_df["split"] == "test") & (sim_df["config"] == "copy_all")].iloc[0]

ttl = 300
if 'Politics' in DEFAULT_TAGS:
    ttl = 20*60

metadata = {
    "type": "scaled_copy",
    "market_filter_mode": MARKET_FILTER_MODE.value,
    "saved_strategy": SAVE_STRATEGY,
    "skip_val_calibration": bool(skip_val),
    "tags": sorted(DEFAULT_TAGS),
    "quote_ttl": ttl,
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": int((contrib["alpha_tier"] > 0).sum()),
    "n_wallets_total": len(wallets),
    "split_sizes": {k: int(len(v)) for k, v in splits.items()},
}

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_scale_sharpe": (None if skip_val
                          else float(max(best_per_scheme.values(), key=lambda x: x[2])[2])),
    "exposure_frac": float(EXPOSURE_FILTER_FRAC),
    "test_performance": {
        "config": best_name,
        "trades": int(test_row["trades"]),
        "pnl": float(test_row["pnl"]),
        "roi_w": float(test_row["roi_w"]),
        "sharpe_daily": float(test_row["sharpe_daily"]),
        "copy_all": {
            "trades": int(copy_all_row["trades"]),
            "pnl": float(copy_all_row["pnl"]),
            "roi_w": float(copy_all_row["roi_w"]),
            "sharpe_daily": float(copy_all_row["sharpe_daily"]),
        },
    },
    "exposure": {
        "frac": float(EXPOSURE_FILTER_FRAC),
        "threshold_type": "wallet_capture_thresholds(fit-window BUY stream, copyable_pnl)",
        "thresholds": {w: float(t) for w, t in exp_th_map.items()},
    },
    "metadata": metadata,
    "wallets": wallet_records,
}

_MODE_RESULT_SUFFIX = {
    MarketFilterMode.DEFAULT: "",
    MarketFilterMode.SPORTS: "_sport",
    MarketFilterMode.CRYPTO: "_crypto",
}
out_path = NB_DIR / f"stage1_scaled_result{_MODE_RESULT_SUFFIX[MARKET_FILTER_MODE]}.json"
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 scaled result -> {out_path.resolve()}")


Saved stage 1 scaled result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_scaled_result_sport.json


## Price-scaling fill experiment (exploratory)

Test a limit-price entry idea on a **sample** (~1k test contracts, copy-default wallets):
copy each candidate copy-wallet BUY at `limit = price * scale` for
`scale ∈ {1.0, 0.98, 0.95, 0.90}` and give the order a **5-minute window** to fill.

- **Fill rule:** filled iff within `(dt, dt+5min]` any trade on the same
  `(condition_id, token_id)` prints at `price <= limit` with a strictly greater timestamp.
- **Fill price:** exactly the limit price, so
  `pnl = copyable_pnl + copyable_qty_5m_100 * (price - limit)` (same formula/quantity as the
  original `copyable_pnl`); unfilled trades contribute 0.
- **Baseline:** `scale = 1.0` is the market-copy (fill immediately at `price`), so it
  must reproduce `sum(copyable_pnl)` on the sample.


In [43]:
from lib import default_trades_dir
from signal_lab.wallet_scaling import price_scale_fill_sim

rng = np.random.RandomState(42)
test_markets = np.sort(splits["test"]["condition_id"].unique())
n_sel = min(1000, len(test_markets))
sel_markets = rng.choice(test_markets, size=n_sel, replace=False)
signals = splits["test"][splits["test"]["condition_id"].isin(sel_markets)].copy()
signals = signals[signals["copyable_qty_5m_100"] > 0]
print(f"test markets: {len(test_markets):,}  sampled: {n_sel:,}")
print(f"candidate BUYs (copyable_qty_5m_100>0) on sample: {len(signals):,}")

_tape_cols = ["condition_id", "token_id", "dt", "avg_price"]
tape_parts = []
for f in sorted(default_trades_dir().glob("*.parquet")):
    tp = pd.read_parquet(f, columns=_tape_cols)
    tp = tp[tp["condition_id"].isin(sel_markets)]
    if not tp.empty:
        tape_parts.append(tp.rename(columns={"avg_price": "price"}))
tape = (
    pd.concat(tape_parts, ignore_index=True)
    if tape_parts
    else pd.DataFrame(columns=["condition_id", "token_id", "dt", "price"])
)
print(f"fill tape rows (sampled contracts, both sides): {len(tape):,}")


test markets: 1,040  sampled: 1,000
candidate BUYs (copyable_qty_5m_100>0) on sample: 2,652
fill tape rows (sampled contracts, both sides): 1,356,592


In [44]:
SCALES = (1.0, 0.98, 0.95, 0.90)
sim = price_scale_fill_sim(signals, tape, scales=SCALES, window_minutes=5.0)
base_pnl = float(signals["copyable_pnl"].sum())

summary = (
    sim.groupby("scale")
    .agg(signals=("filled", "size"), fills=("filled", "sum"),
         fill_rate=("filled", "mean"), pnl=("pnl", "sum"))
    .reset_index()
)
summary["pnl_pct_of_market"] = summary["pnl"] / base_pnl * 100 if base_pnl else np.nan
summary["delta_vs_market"] = summary["pnl"] - base_pnl
print(f"market-copy pnl (baseline = sum copyable_pnl): {base_pnl:,.2f}")
summary.round(2)


market-copy pnl (baseline = sum copyable_pnl): 196,272.86


,scale,signals,fills,fill_rate,pnl,pnl_pct_of_market,delta_vs_market
0,0.9000,2652,1188,0.4500,120747.3100,61.5200,-75525.5400
1,0.9500,2652,1493,0.5600,201221.9800,102.5200,4949.1200
2,0.9800,2652,1732,0.6500,211103.1500,107.5600,14830.2900
3,1.0000,2652,2652,1.0000,196272.8600,100.0000,0.0000


In [45]:
pw_pnl = sim.pivot_table(index="wallet", columns="scale", values="pnl", aggfunc="sum")
pw_fill = sim.pivot_table(index="wallet", columns="scale", values="filled", aggfunc="mean")
pw = pw_pnl.join(pw_fill.rename(columns={c: f"fill_{c:g}" for c in pw_fill.columns}))
pw = pw.reindex(pw[1.0].sort_values(ascending=False).index)
pw.round(1).head(15)


scale,0.9000,0.9500,0.9800,1.0000,fill_0.9,fill_0.95,fill_0.98,fill_1
wallet,,,,,,,,
0x7986576408b65a9805ac85553be33ad0b044baa5,57427.2000,45885.9000,50447.1000,58266.7000,0.1000,0.1000,0.1000,1.0000
0xf3ce7f04dde4f8c5aed19a20e5ecd8520e5ca57a,-11169.7000,43731.7000,65505.5000,47328.3000,0.3000,0.6000,0.9000,1.0000
0xc3973025163de81e367ad29c21e296e51684bb3b,27042.5000,44995.5000,45872.8000,37894.6000,0.6000,0.7000,0.8000,1.0000
0x08c263965a9cbec5e83ac1d6f14b9bbc957bc023,9554.6000,50246.1000,43306.9000,37723.4000,0.6000,0.7000,0.8000,1.0000
0x58f8f1138be2192696378629fc9aa23c7910dc70,27304.8000,10912.9000,12559.8000,11485.5000,0.2000,0.4000,0.7000,1.0000
0x5c2311acccc5e9a172f0ce23d4480177bc00cab3,6265.0000,6128.7000,5863.7000,8670.2000,0.2000,0.4000,0.4000,1.0000
0x64f4577177d590e78174b0c5364af8e80fbab38e,6446.3000,12799.3000,7745.3000,7916.0000,0.5000,0.6000,0.8000,1.0000
0x795173772d871733c3110a5695d9638e4b50bb41,808.9000,2283.6000,5825.3000,4755.2000,0.0000,0.2000,0.3000,1.0000
0xd6a39cef30c927c4d57b5cc519f3a2d0332a9738,0.0000,0.0000,3291.4000,3760.2000,0.0000,0.0000,0.3000,1.0000


In [46]:
# sim.to_csv(OUT_DIR / "price_scale_sim.csv", index=False)
# summary.round(4).to_csv(OUT_DIR / "price_scale_summary.csv", index=False)
# pw.round(2).reset_index().to_csv(OUT_DIR / "price_scale_wallets.csv", index=False)
# print("saved -> signal_lab/price_scale_{sim,summary,wallets}.csv")
